### Redes Neurais Recorrentes (RNNs)

Diferente de sistemas cognitivos ou processamentos sequenciais naturais, em que o contexto anterior é continuamente utilizado para interpretar novos estímulos, as redes neurais tradicionais não possuem um mecanismo nativo para retenção de estado. Elas processam cada entrada de forma isolada, o que representa uma desvantagem em tarefas que dependem de ordem cronológica. Por exemplo, uma rede tradicional seria incapaz de utilizar *features* extraídas de eventos passados de um vídeo para inferir o próximo estado. As Redes Neurais Recorrentes (*Recurrent Neural Networks*, RNNs) resolvem esse problema implementando retroalimentação em sua estrutura. Esse design permite a propagação de variáveis de estado ao longo de instantes de tempo $t$, dotando a rede da capacidade de mapear dependências temporais.

<figure style="text-align: center;">
  <img src="assets/figura-1.png" alt="Figura 1" width="800">
</figure>

**Figura 1** – Diagrama de uma RNN (bloco A) e sua forma aplicada em uma cadeia.

Este tipo de arquitetura permite que a informação seja propagada de um módulo da rede para o seguinte. Uma RNN pode ser visualizada como múltiplas cópias da mesma rede, onde cada iteração repassa o seu vetor de estado para a sucessora. Essa estrutura em forma de cadeia torna as RNNs uma arquitetura ideal para lidar com dados sequenciais, sendo amplamente aplicadas em tarefas como reconhecimento de fala, modelagem de linguagem, tradução automática e legendagem de imagens.

### Importância das Dependências de Longo Prazo

A vantagem de uso das RNNs reside na capacidade de mapear informações passadas para a tarefa de inferência atual. Em cenários onde a dependência é de curto prazo, o modelo necessita apenas de observações recentes. Por exemplo, na sequência "as nuvens estão no...", a predição da palavra "céu" exige apenas o contexto imediato. Nesses casos, o intervalo temporal entre a informação prévia e o instante em que ela é necessária é pequeno, permitindo que arquiteturas RNN tradicionais apresentem um desempenho adequado.

<figure style="text-align: center;">
  <img src="assets/figura-2.png" alt="Figura 2" width="400">
</figure>

**Figura 2** – Diagrama de uma RNN mostrando uma dependência de curto prazo.

Por outro lado, a complexidade aumenta ao considerarmos o trecho: "Eu cresci na França [... longos textos ...] Eu falo francês fluentemente." A presença do termo recente "falo" restringe a predição a um idioma, mas para determinar o rótulo correto ("francês"), a rede precisa recuperar a informação espacial ("França"), que se encontra consideravelmente mais distante na sequência temporal em relação ao exemplo anterior.

<figure style="text-align: center;">
  <img src="assets/figura-3.png" alt="Figura 3" width="400">
</figure>

**Figura 3** – Diagrama de uma RNN mostrando uma dependência de longo prazo.

À medida que essa lacuna entre a informação relevante e o ponto de uso aumenta, as RNNs clássicas tornam-se ineficazes no aprendizado dessas correlações. Embora exista a sustentação teórica para que lidem com dependências de longo prazo, a otimização na prática falha. Desse modo, as arquiteturas LSTM (*Long Short-Term Memory*) foram concebidas especificamente para mitigar essa limitação, solucionando o problema do desvanecimento do gradiente (*vanishing gradient problem*) e viabilizando a retenção de informações em longas sequências. A seguir, vamos entender melhor os artifícios usados para solucionar tal problemática.

## Redes LSTM (Long Short-Term Memory)

As LSTMs representam uma classe especializada de RNNs, otimizada especificamente para o aprendizado de dependências de longo prazo. Introduzidas formalmente por Hochreiter e Schmidhuber (1997), essas redes consolidaram-se como a arquitetura fundamental por trás dos avanços mais significativos no processamento de dados sequenciais.

<figure style="text-align: center;">
  <img src="assets/figura-4.png" alt="Figura 4" width="400">
</figure>

**Figura 4** – Sepp Hochreiter e Jürgen Schmidhuber, pesquisadores responsáveis pela proposição da arquitetura LSTM.

As LSTMs foram estruturalmente projetadas para solucionar a falha das RNNs clássicas, mitigando o problema do desvanecimento do gradiente em sequências extensas. Dessa forma, a capacidade de reter informações ao longo de múltiplos instantes de tempo é um comportamento inerente à sua própria arquitetura.

### A Arquitetura de uma LSTM

A célula recorrente em uma arquitetura RNN padrão possui uma topologia elementar, consistindo tipicamente de uma única camada neural dotada de uma função de ativação `tanh`. A cada instante de tempo, ele recebe a nova informação $(x_{t})$ e junta com a memória vinda do passo anterior $(h_{t−1})$. Essa combinação passa pela função $tanh$, que apenas ajusta os valores para uma escala padrão entre −1 e 1, gerando o novo estado oculto $(h_{t})$. A limitação dessa estrutura é que, por ter apenas essa operação simples, a rede é obrigada a reescrever toda a sua memória a cada passo, sem um mecanismo para selecionar o que deve ser mantido ou descartado.

<figure style="text-align: center;">
  <img src="assets/figura-5.png" alt="Figura 5" width="600">
</figure>

**Figura 5** – O módulo repetitivo em uma RNN padrão com uma única camada tanh.

A LSTM preserva o princípio da estrutura em cadeia temporal. No entanto, o seu módulo repetitivo (célula) apresenta um estruturação mais sofisticada. Em vez de uma única transformação neural, a célula LSTM abriga quatro camadas interativas estruturadas de forma a controlar o fluxo de informação.

<figure style="text-align: center;">
  <img src="assets/figura-6.png" alt="Figura 6" width="600">
</figure>

**Figura 6** – O módulo repetitivo em uma LSTM com quatro camadas interagindo.

O componente central da arquitetura LSTM é o **Estado da Célula** (*Cell State*). Representado visualmente pela linha horizontal superior nos diagramas, ele atua conceitualmente como uma "esteira transportadora" de dados. O estado da célula atravessa toda a cadeia temporal sujeito primariamente a operações lineares elementares. Essa característica arquitetural é o que permite que a informação (e o gradiente) flua através do tempo sem sofrer alterações drásticas ou atenuação severa.

<figure style="text-align: center;">
  <img src="assets/figura-7.png" alt="Figura 7" width="400">
</figure>

**Figura 7** – O estado da célula (linha horizontal superior) fluindo de $C_{t-1}$ (informação passada) para $C_t$ (informação presente).

A rede possui a capacidade de adicionar ou remover dinamicamente informações do estado da célula por meio de estruturas regulatórias denominadas **portas** (*gates*). Esses *gates* funcionam como filtros que permitem a passagem opcional da informação. Estruturalmente, um *gate* é composto por uma camada densa com função de ativação `sigmoide` combinada a uma operação de multiplicação elemento a elemento (*element-wise*).

<figure style="text-align: center;">
  <img src="assets/figura-8.png" alt="Figura 8" width="200">
</figure>

**Figura 8** – Diagrama de uma porta (gate) com função de ativação sigmoide.

A função sigmoide mapeia os dados de entrada para o intervalo restrito de $0$ a $1$. Nesse contexto, o valor $0$ indica "bloqueio total da informação", enquanto o valor $1$ sinaliza "permissão total de passagem". Uma célula LSTM padrão implementa três desses *gates* para proteger e modular de forma precisa o estado da célula.

<figure style="text-align: center;">
  <img src="assets/figura-9.png" alt="Figura 9" width="500">
</figure>

**Figura 9** – Função sigmoide.

Antes de detalharmos a atuação matemática específica de cada *gate* sobre o estado da célula, é imperativo estabelecer a notação arquitetural dos diagramas. A compreensão prévia dessa simbologia facilitará a análise das equações e operações subsequentes:

<figure style="text-align: center;">
  <img src="assets/figura-10.png" alt="Figura 10" width="500">
</figure>

**Figura 10** – Notações matemáticas e esquemáticas do diagrama.

### Portão 1 - O Portão do Esquecimento (*Forget Gate*)

O primeiro passo na operação de uma LSTM é decidir quais informações da memória passada ($C_{t-1}$) devem ser mantidas ou descartadas. Essa decisão é realizada por uma camada interna dotada de uma função de ativação `sigmoide`, denominada camada da porta de esquecimento (*forget gate layer*, representada por $f_t$). 

Ela processa o estado oculto anterior ($h_{t-1}$) em conjunto com a nova entrada temporal ($x_t$). Como resultado, gera um vetor onde cada componente assume um valor entre 0 (que significa "descarte completamente") e 1 (que significa "retenha completamente") para atuar sobre o estado da célula $C_{t-1}$.

Para ilustrar, imagine uma tarefa de predição de texto: o estado da célula pode estar armazenando o gênero de um sujeito para garantir a concordância correta do próximo pronome. Ao encontrar um novo sujeito na frase, a porta de esquecimento atua instruindo a rede a apagar a informação do gênero do sujeito antigo, liberando espaço para o novo contexto.

<figure style="text-align: center;">
  <img src="assets/figura-11.png" alt="Figura 11" width="600">
</figure>

**Figura 11** – A porta de esquecimento (forget gate). O vetor de ativação $f_t$ é calculado aplicando-se a função sigmoide ($\sigma$) à concatenação do estado oculto anterior ($h_{t-1}$) com a entrada atual ($x_t$), parametrizada pela matriz de pesos $W_f$ e pelo vetor de viés $b_f$.

### Portão 2 - O Portão de Entrada (*Input Gate*)

O próximo passo é decidir quais novas informações serão armazenadas no estado da célula. Esta etapa opera por meio de duas partes complementares:

* **(a)** Uma camada interna com função de ativação `sigmoide` (a camada da porta de entrada, ou *input gate layer*, $i_t$) decide quais valores da memória serão atualizados.
* **(b)** Simultaneamente, uma camada com ativação `tanh` cria um vetor de novos valores candidatos ($\tilde{C}_t$), que representam a informação potencial a ser adicionada ao estado.

Dando continuidade ao exemplo de processamento de texto, é nesta etapa que a rede identifica e prepara a informação sobre o gênero do *novo* sujeito para ser incorporada ao estado da célula, substituindo o dado antigo que a porta de esquecimento descartou.

<figure style="text-align: center;">
  <img src="assets/figura-12.png" alt="Figura 12" width="600">
</figure>

**Figura 12** – A porta de entrada (input gate) e a camada de geração de valores candidatos. O vetor de ativação $i_t$ é calculado aplicando-se a função sigmoide ($\sigma$) à concatenação de $h_{t-1}$ e $x_t$, com seus parâmetros $W_i$ e $b_i$. Simultaneamente, o vetor candidato $\tilde{C}_t$ é gerado aplicando-se a função $\tanh$ à mesma concatenação, utilizando uma matriz de pesos ($W_C$) e um viés ($b_C$) independentes.

Nesta etapa, a arquitetura consolida as decisões computadas anteriormente para atualizar o estado antigo da célula ($C_{t-1}$) para o novo estado ($C_t$). A transição ocorre por meio das seguintes operações algébricas:

* **(a)** O estado anterior ($C_{t-1}$) é multiplicado, elemento a elemento, pelo vetor da porta de esquecimento ($f_t$). Essa operação executa matematicamente o descarte das informações julgadas irrelevantes (quando os valores de $f_t$ tendem a 0) ou a manutenção do contexto (quando tendem a 1).
* **(b)** Em seguida, adiciona-se o produto elemento a elemento entre o vetor da porta de entrada ($i_t$) e o vetor de valores candidatos ($\tilde{C}_t$). Essa etapa insere as novas informações na memória, ponderadas pelo grau de relevância que a rede atribuiu a cada componente.

Retomando o exemplo de modelagem de linguagem, esta é a etapa computacional em que a rede efetivamente apaga a informação do gênero do sujeito anterior da sua memória e consolida a informação do novo sujeito no estado atual da célula.

<figure style="text-align: center;">
  <img src="assets/figura-13.png" alt="Figura 13" width="600">
</figure>

**Figura 13** – A atualização do estado da célula. O novo estado ($C_t$) é obtido somando-se a memória passada filtrada ($f_t \odot C_{t-1}$) à nova informação candidata ponderada ($i_t \odot \tilde{C}_t$).

### Portão 3 - O Portão de Saída (*Output Gate*)

Por fim, a arquitetura determina qual informação será propagada como o estado oculto atual ($h_t$). Essa saída é derivada do estado da célula recém-atualizado ($C_t$), porém submetida a um processo de filtragem estruturado nos seguintes passos:

* **(a)** Uma camada densa com função de ativação `sigmoide` (a porta de saída, ou *output gate layer*, $o_t$) processa a entrada atual e o estado oculto anterior para definir quais componentes do estado da célula serão expostos.
* **(b)** O estado atualizado da célula ($C_t$) é submetido diretamente à função matemática `tanh`, projetando os seus valores para o intervalo restrito de $[-1, 1]$.
* **(c)** O resultado dessa projeção é multiplicado, elemento a elemento, pelo vetor de ativação da porta de saída ($o_t$).

Concluindo o exemplo de modelagem de linguagem: após a rede consolidar a presença de um novo sujeito na sua memória interna, a porta de saída atua filtrando esse estado para propagar apenas as características sintáticas imediatas, como o número (singular ou plural) do sujeito. Dessa forma, garante-se que a inferência do verbo subsequente receba o contexto adequado para ser julgada corretamente, sem que a rede exponha toda a sua memória interna desnecessariamente.

<figure style="text-align: center;">
  <img src="assets/figura-14.png" alt="Figura 14" width="600">
</figure>

**Figura 14** – A porta de saída (*output gate*) e o cálculo final do estado oculto. O vetor $o_t$ define o filtro por meio da ativação sigmoide ($\sigma$), enquanto o novo estado oculto ($h_t$) é obtido pelo produto de elemento a elemento entre $o_t$ e a tangente hiperbólica do estado da célula atual ($\tanh(C_t)$).

Dessa forma, completamos o ciclo computacional de uma célula LSTM funcional. Em retrospectiva, nota-se que o vetor de valores candidatos ($\tilde{C}_t$) é o único componente responsável por extrair e gerar novas propostas de informação a partir da entrada atual. Em contrapartida, as ativações $f_t$, $i_t$ e $o_t$ operam estritamente como mecanismos de regulação (filtros), determinando, respectivamente, o que deve ser mantido da memória histórica, o que deve ser incorporado do contexto presente e o que deve ser propagado como a saída da rede.

| **Notação** | **Componente** | **Função Arquitetural** |
| :--- | :--- | :--- |
| **$\tilde{C}_t$** | Valores Candidatos | Extrair e gerar novas propostas de informações a partir da entrada atual e do estado oculto anterior. |
| **$f_t$** | Porta de Esquecimento (*Forget Gate*) | Regular a proporção exata da memória passada ($C_{t-1}$) que será retida ou descartada. |
| **$i_t$** | Porta de Entrada (*Input Gate*) | Controlar qual fração das novas informações candidatas ($\tilde{C}_t$) será, de fato, incorporada à memória. |
| **$o_t$** | Porta de Saída (*Output Gate*) | Filtrar o estado da célula atualizado ($C_t$) para definir o novo estado oculto ($h_t$) emitido pela rede. |

## Próximas Etapas de Aprendizagem

Sinta-se a vontade para visualizar o conteúdo que usamos como referência nesse laboratório para continuar seus estudos sobre LSTM, o material está disponível em [colah's blog](https://colah.github.io/posts/2015-08-Understanding-LSTMs/), na qual também é falado sobre outras variantes mais robustas do LSTM. Uma versão em .pdf desse material também está disponível para acesso e disponibilizado junto com os arquivos de aula.

## Aplicação - Classificação de Sentimentos com LSTM

Nosso objetivo com essa tarefa é treinar um modelo para ler resenhas de filmes (com escrita em inglês) e classificá-la como positiva (nota 1) ou negativa (nota 0). Para isso, usamos o [dataset do IMDB](https://ai.stanford.edu/~amaas/data/sentiment/), que contém 50,000 resenhas de filmes rotuladas por sentimento. Isso é um problema fundamentalmente sequencial. O sentimento de uma resenha não depende apenas das palavras individuais, mas da ordem e do contexto em que elas aparecem. Vamos pegar três contextos como problema:

* (a) O filme é bom!
* (b) O filme é ruim!
* (c) No inicio, eu achei que seria um filme terrível, mas conforme a história foi se desencadeando, fui cativado e achei o final perfeito!

Os casos (a) e (b) são relativamente fáceis de lidar, mas o caso (c), para classificar essa resenha como positiva, o modelo precisa lembrado do contexto de inicio (terrível), entender a transição (mas) e dar peso correto ao final (perfeito). Como vimos na teoria, as Redes Neurais Recorrentes (RNNs) padrão têm dificuldade em manter contexto por longos períodos (o problema de mémoria de longo prazo) Entretanto, os LSTMs conseguem lidar melhor com esse problema, pois:

* Mantém informações relevantes.
* Esquecem informações irrelevantes.
* Atualizam o seu entendimento a medida que lê a sequência.

<figure style="text-align: center;">
  <img src="assets/figura-15.png" alt="Figura 15" width="400">
</figure>

**Figura 15** – Avaliações do IMDB. Um exemplo de como as resenhas são escritas.

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 1:</b><br>
  Para o exemplo apresentado na Figura 15, tentando entender como um ser humano faz essa avaliação, qual nota de 0 a 10 você daria, considerando 10 como muito positivo e 0 como muito negativo?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>


In [ ]:
# !pip install scikit-learn seaborn plotly

In [ ]:
from tensorflow.keras.layers import Embedding, LSTM, Dense, SimpleRNN
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.datasets import imdb
import tensorflow as tf

from sklearn.metrics import confusion_matrix
import plotly.graph_objects as go
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re

import plotly.io as pio

pio.renderers.default = "notebook"

### Análise dos Dados

In [ ]:
vocab_size = 10000 # tamanho do vocabulário (palavras mais comuns)
max_length = 500 
embedding_dim = 32

In [ ]:
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)

In [ ]:
print(f"Total de amostras de treino {len(y_train)}")
print(f"Avaliações Positivas (1): {np.sum(y_train == 1)}")
print(f"Avaliações Negativas (0): {np.sum(y_train ==0)}")

In [ ]:
print(f"Total de amostras de teste {len(y_test)}")
print(f"Avaliações Positivas (1): {np.sum(y_test == 1)}")
print(f"Avaliações Negativas (0): {np.sum(y_test ==0)}")

Vamos carregar os dados do *dataset*, limitando a quantidade de palavras baseado na variável `vocab_size`. Um ponto de dúvida é que as resenhas tem tamanhos variáveis, então o que significa o parâmetro `num_words`, que limita a quantidade de palavras? 

O *dataset* IMDB do Keras contém resenhas de filmes representadas como sequências de índices inteiros, onde cada número corresponde a uma palavra, por exemplo [14, 20, 43, 2, 500, ..., 17]. Cada número é o ID da palavra no vocabulário.

O parâmetro `num_words` não limita o tamanho das resenhas, e sim o tamanho do vocabulário (ou seja, quantas palavras diferentes o modelo vai considerar). As palavras que não estão no top-10k são substituidas por um token especial `<UNK>`, que tem índice 2. Então porque não simplesmente importar todas as palavras? A resposta é: depende da sua aplicação. No nosso objetivo, de avaliar as resenhas como positivas ou negativas usando uma RNN com LSTMs em suas camadas, limitar a quantidade de palavras tem por objetivo:

* reduz a dimensão do embedding e a memória usada.
* evita ruiído de palavras raras (pouco informativas).
* acelera o treino e reduz o *overfitting*.

A seguir, vamos analisar melhor o nosso conjunto de dados.

In [ ]:
(x_train_orig, _), _ = imdb.load_data(num_words=vocab_size)

review_lengths = [len(review) for review in x_train_orig]

print(f"Comprimento médio da resenha: {np.mean(review_lengths):.2f} palavras.")
print(f"Comprimento máximo: {np.max(review_lengths)} palavras.")
print(f"Comprimento mínimo: {np.min(review_lengths)} palavras.")

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(review_lengths, bins=50, color="orange")
plt.title("Distribuição do Comprimento das Resenha (Palavras)", fontsize=14)
plt.xlabel("Número de Palavras")
plt.ylabel("Número de resenhas")

plt.axvline(max_length, color="red", linestyle="--", linewidth=2,
            label=f"Ponto de Corte ({max_length} palavras)")

plt.legend()
plt.show()

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 2:</b><br>
  Você imagina que há relação entre o comprimento da resenha e a nota atribuída?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

Agora, vale a pena conhecer o formato em linguagem natural das resenhas, a seguinte célula atribui a cada ID das palavras (ou *embeddings*) por seus respectivos rótulos.

In [ ]:
word_index = imdb.get_word_index()

# os índices 0, 1, 2 e 3 são reservados para <PAD>, <START>, <UNK>, <UNUSED>
reverse_word_index = {v + 3: k for k, v in word_index.items()}
reverse_word_index[0] = "<PAD>"
reverse_word_index[1] = "<START>"
reverse_word_index[2] = "<UNK>"
reverse_word_index[3] = "UNUSED"

def decode_review(sequence):
    # converte uma sequência de índices de volta ao texto
    return ' '.join([reverse_word_index.get(i, "?") for i in sequence])

print(decode_review(x_train_orig[0]))

Agora, sabemos que as sequências tem tamanhos variáveis, vamos padronizá-las para que tenham tamanhos fixos para que nosso modelo enxergue um tensor 2D uniforme para processar, na qual iremos ter dois efeitos aplicados:

* Trunca sequências muito longas (corta o excesso).
* Preenche (*padding*) as curtas com zeros.

In [ ]:
# padroniza o comprimento das sequências
x_train = pad_sequences(x_train, maxlen=max_length)
x_test = pad_sequences(x_test, maxlen=max_length)

In [ ]:
plt.figure(figsize=(12, 8))
examples_limit = 20
sns.heatmap(x_train[:examples_limit], cmap="viridis", cbar=True)

plt.title(f"Visualização das 20 primeiras resenhas (Padronizadas para {max_length} palavras)", fontsize=14)
plt.xlabel(f"Palavras (índices de 0 a {max_length-1})")
plt.ylabel(f"Resenhas (índices de 0 a {examples_limit-1})")
plt.show()

Se o preenchimento fosse feito à direita (*post-padding*), a rede processaria a mensagem real nos primeiros passos de tempo e, em seguida, seria forçada a iterar sobre centenas de valores nulos (zeros), o que causaria degradação e esquecimento do estado oculto $(h_{t})$ antes de gerar a predição final. Então, devido a limitações arquiteturais, o preenchimento foi feito à esquerda (*pre-padding*).

### Modelo com RNN Simples

Antes de partirmos para um exploração envolvendo blocos de LSTM, vamos criar um modelo com camadas de RNN simples. que servirá como referencial. Tal modelo com blocos de RNN simples, é útil para processamento de textos sequenciais, mas não possui mecanismos internos de "memória de longo prazo". Ou seja, devido ao desvanecimento do gradiente, acaba esquecendo informações contextuais mais antigas à medida que lê novas palavras.

Mais a frente, nosso objetivo será comparar seu desempenho com a LSTM e verificar empiricamente como a RNN falha em capturar dependências longas em textos mais complexos.

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 3:</b><br>
  Você imagina que um modelo com camadas de RNN simples vai ter um desempenho melhor ou pior que um modelo com camadas de LSTM? Explique seu ponto de vista e relacione o problema do desvanecimento do gradiente com os tamanhos da resenha no geral.
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

In [ ]:
model_rnn = Sequential([
    Embedding(vocab_size, embedding_dim),
    SimpleRNN(64, return_sequences=True),
    SimpleRNN(32),
    Dense(1, activation='sigmoid')
])

model_rnn.compile(optimizer="adam", loss="binary_crossentropy",
                  metrics=["accuracy"])

model_rnn.build(input_shape=(None, max_length))
model_rnn.summary()

In [ ]:
early_stopping = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
history_rnn = model_rnn.fit(
    x_train, y_train, epochs=10, batch_size=128, validation_split=0.2, 
    callbacks=[early_stopping], verbose=True
)

In [ ]:
# extração da penúltima camada do modelo
model_input = model_rnn.layers[0].input
rnn_layer_output = model_rnn.layers[2].output

# cria um modelo auxiliar que retorna as features intermediárias
feature_extractor_rnn = Model(inputs=model_input, outputs=rnn_layer_output)

# seleciona amostras aleatórias do conjunto de teste
num_samples_to_plot = 1000
np.random.seed(42)
indices = np.random.choice(len(x_test), num_samples_to_plot, replace=False)
x_test_sample = x_test[indices]
y_test_sample = y_test[indices]

rnn_features = feature_extractor_rnn.predict(x_test_sample, verbose=0)

# prevê os sentimentos dessas amostras
y_pred_probs_rnn = model_rnn.predict(x_test_sample, verbose=0)
y_pred_rnn = (y_pred_probs_rnn > 0.5).astype(int).flatten()

results = np.where(y_pred_rnn == y_test_sample, "Correto", "Erro")

# reduz as features para 3 dimensões com t-SNE
tsne_rnn = TSNE(n_components=3, random_state=42, perplexity=30, max_iter=1000)
features_3d_rnn = tsne_rnn.fit_transform(rnn_features)

colors = []
for true, pred in zip(y_test_sample, y_pred_rnn):
    if true == 0 and pred == 0: colors.append("blue")      
    elif true == 1 and pred == 1: colors.append("green")      
    elif true == 0 and pred == 1: colors.append("red")       
    else: colors.append("orange")     

hover_labels = [
    f"Verdadeiro: {'Positivo' if t == 1 else 'Negativo'} | "
    f"Previsto: {'Positivo' if p == 1 else 'Negativo'}"
    for t, p in zip(y_test_sample, y_pred_rnn)
]

In [ ]:
# plot interativo 3D
fig = go.Figure(data=[go.Scatter3d(
    x=features_3d_rnn[:, 0],
    y=features_3d_rnn[:, 1],
    z=features_3d_rnn[:, 2],
    mode='markers',
    marker=dict(
        size=5,
        color=colors,
        opacity=0.8
    ),
    text=hover_labels,
    hoverinfo='text'
)])

fig.update_layout(
    title="Visualização 3D — Modelo com RNN Simples",
    scene=dict(
        xaxis_title='Componente t-SNE 1',
        yaxis_title='Componente t-SNE 2',
        zaxis_title='Componente t-SNE 3'
    ),
    margin=dict(r=20, l=10, b=10, t=40),
    legend_title_text="Classificação"
)

fig.show()

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 4:</b><br>
  Observando a projeção 3D das representações obtidas pela RNN, as amostras de diferentes classes formam agrupamentos distintos?
O que essa separação (ou sobreposição) indica sobre a capacidade do modelo em diferenciar os padrões dos dados?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>


In [ ]:
loss_lstm, acc_lstm = model_rnn.evaluate(x_test, y_test)

In [ ]:
y_pred_probs = model_rnn.predict(x_test)

y_pred = (y_pred_probs > 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Negativo", "Positivo"],
            yticklabels=["Negativo", "Positivo"])

plt.title("Matriz de Confusão")
plt.ylabel("Rótulo Verdadeiro")
plt.xlabel("Rótulo Previsto")

plt.show()

### Modelo LSTM

A seguir, vamos implementar um modelo LSTM, utilizado para analisar as resenhas que carregamos a partir do *dataset* do IMDB. Ele aprende padrões nas sequências de palavras para classificar automaticamente cada *review* como positiva ou negativa. O modelo, é separado em três camadas:

* **Camada de *Embedding***: Transforma os índices das palavras em vetores densos (representação vetorial)
* **Camada LSTM**: Captura relações de longo prazo na sequência, resolvendo o problema do "desvenencimento do gradiente", algo que seria comum caso fosse aplicado uma camada de RNN simples. Uma camada aparentemente ideal para textos longos como as nossas resenhas.
* **Camada de Saída**: produz uma probabilidade entre 0 e 1 (sentimento negativo ou positivo).

O modelo e didaticamente simples, sinta-se a vontade para mexer nos hiperparâmetros e expandir as camadas do modelo e veja sua performace nos dados de teste melhorando (ou piorando).

In [ ]:
model_lstm = Sequential([
    Embedding(vocab_size, embedding_dim),
    LSTM(64, return_sequences=True),
    LSTM(32),
    Dense(1, activation='sigmoid')
])

model_lstm.compile(optimizer='adam',
                   loss='binary_crossentropy',
                   metrics=['accuracy'])

model_lstm.build(input_shape=(None, max_length)) # Forçar um exemplo para buildar
model_lstm.summary()

In [ ]:
early_stopping = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

history_lstm = model_lstm.fit(x_train, y_train, epochs=10, batch_size=128, validation_split=0.2,
                              callbacks=[early_stopping]) 

Vamos analisar como o modelo está performando nos dados de teste.

In [ ]:
loss_lstm, acc_lstm = model_lstm.evaluate(x_test, y_test)

In [ ]:
def plot_training_history(history, title):
    # plota os gráficos de acurácia e perda a partir do histórico de treinamento
    acc = history.history["accuracy"]
    val_acc = history.history["val_accuracy"]
    loss = history.history["loss"]
    val_loss = history.history["val_loss"]
    
    epochs = range(1, len(acc) + 1)

    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, acc, "bo-", label="Acurácia de Treino")
    plt.plot(epochs, val_acc, 'ro-', label="Acurácia de Validação")
    plt.title(f'Acurácia de Treino e Validação - {title}')
    plt.xlabel("Épocas")
    plt.ylabel("Acurácia")
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss, "bo-", label="Perda de Treino")
    plt.plot(epochs, val_loss, "ro-", label="Perda de Validação")
    plt.title(f"Perda de Treino e Validação - {title}")
    plt.xlabel("Épocas")
    plt.ylabel("Perda")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

plot_training_history(history_lstm, "LSTM")


<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 5:</b><br>
  Observando as curvas de acurácia e perda do modelo LSTM, é possível identificar indícios de overfitting?
Em que ponto (época) o modelo parece começar a perder capacidade de generalização?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

In [ ]:
y_pred_probs = model_lstm.predict(x_test)

y_pred = (y_pred_probs > 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Negativo", "Positivo"],
            yticklabels=["Negativo", "Positivo"])

plt.title("Matriz de Confusão")
plt.ylabel("Rótulo Verdadeiro")
plt.xlabel("Rótulo Previsto")

plt.show()

In [ ]:
# dicionário de palavras
word_index = imdb.get_word_index()

# o keras reserva os 3 primeiros índices para <PAD>, <START>, <UNK>
word_index = {k:(v+3) for k, v in word_index.items()}
word_index["<PAD>"] = 0
word_index["<START>"] = 1
word_index["<UNK>"] = 2  
word_index["<UNUSED>"] = 3

def preprocess_text(text, word_index, max_length, vocab_size):
    # converte uma string de texto em uma sequência de tokens pronta para o modelo
    text = text.lower()
    # remove pontuação (caractere que não seja letra, número ou espaço)
    text = re.sub(r"[^a-zA-Z0-9\s]", '', text) 
    
    # tokenização (palavras convertidas em embeddings)
    tokens = []
    for word in text.split():
        token = word_index.get(word, 2) # 2 é o token <UNK>
        
        if token < vocab_size:
            tokens.append(token)
        else:
            tokens.append(2) # <UNK>
            
    # adiciona o token <START> no início
    tokens = [1] + tokens
            
    # padronização do comprimento (padding)
    padded_tokens = pad_sequences([tokens], maxlen=max_length, padding="pre")
    
    return padded_tokens

def predict_sentiment(text):
    # usa o modelo LSTM treinado para prever o sentimento de um texto.
    global model_lstm, word_index, max_length, vocab_size

    # pré-processamento do texto
    processed_text = preprocess_text(text, word_index, max_length, vocab_size)

    # predição no texto de análise
    prediction_prob = model_lstm.predict(processed_text)[0][0]
    
    print(f"\nTexto: '{text}'")
    print(f"Probabilidade: {prediction_prob:.4f}")

In [ ]:
predict_sentiment("This movie was absolutely fantastic, the acting was great and the plot was amazing.")
predict_sentiment("I hated this film. It was boring, slow, and a complete waste of time.")
predict_sentiment("The visual effects were incredible and the sound design was top tier, but the story was empty and boring.")
predict_sentiment("I really wanted to love this movie. The trailer looked like a masterpiece, but the actual film was a huge letdown.")
predict_sentiment("If you are a fan of mindless action with zero plot, this is perfect. Otherwise, stay away.")
predict_sentiment("Oh, brilliant. Another superhero movie with the exact same plot as the last fifty ones. Just what we needed.")
predict_sentiment("It was not the best movie I have ever seen, but it was certainly not the worst. It was just okay.")
predict_sentiment("I cannot say I didn't enjoy it. It was actually better than I expected.")
predict_sentiment("At first I thought this movie was going to be terrible, but the ending was absolutely perfect.")
predict_sentiment("Funny at times, yet I wouldn't watch it again.")
predict_sentiment("Nice visuals and great performances, but the plot was completely confusing, waste of time and terrible.")

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 6:</b><br>
  Observando os resultados do modelo LSTM para frases ambíguas, por que ele tende a classificar algumas sentenças contraditórias de forma equivocada?
O que isso revela sobre as limitações do modelo em compreender contexto e ironia?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 7:</b><br>
  O modelo parece dar mais peso às palavras negativas ou positivas?
Como isso pode afetar a precisão quando a frase mistura os dois tons? Teste com novas frases (em inglês) e avalie esse comportamento.
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

A seguir, vamos trabalhar em uma visualização 3D que nos permite inspecionar o que a rede LSTM aprendeu sobre o significado das palavras. Cada ponto no gráfico representa uma palavra, cuja posição é determinada pelo seu *embedding*, que é o vetor número que o modelo aprendeu para ela. Como os *embeddings* originais têm muitas dimensões, especificamente 32D, usamos a técnica de t-SNE para "comprimir" esses vetores em 3D, tentando ao máximo preservar suas distâncias relativas.

O objetivo é ver se o modelo agrupou palavras de significado similar, algo que forme um *cluster* para palavras de sentimento positivo, negativo e neutro. Isso é um forte indicador de que o modelo conseguiu aprender a associar palavras com seus respectivos sentimentos.

In [ ]:
positive_words = [
    "good", "great", "amazing", "perfect", "love", "best", "fantastic",
    "excellent", "wonderful", "beautiful"
]

negative_words = [
    "bad", "terrible", "awful", "waste", "boring", "hated", "worst",
    "stupid", "poor", "sad"
]
neutral_words = [
    "movie", "film", "actor", "actors", "story", "plot", "it", "is",
    "the", "a", "this", "but", "not"
]

word_groups = {
    "green": positive_words,  
    "red": negative_words,   
    "yellow": neutral_words   
}

# Intercepta os pesos aprendidos no meio do modelo
embedding_layer = model_lstm.layers[0]
embedding_weights = embedding_layer.get_weights()[0]

vectors = []
labels = []
colors = []  

for color, word_list in word_groups.items():
    for word in word_list:
        idx = word_index.get(word)
        # Verifica se a palavra existe no vocabulário
        if idx is not None and idx < vocab_size:
            vectors.append(embedding_weights[idx])
            labels.append(word)
            colors.append(color)  #


vectors = np.array(vectors)

# Reduz para um dimensionalidade 3D
tsne = TSNE(n_components=3, random_state=42, perplexity=15,
            max_iter=1000)
vectors_3d = tsne.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter3d(
    x=vectors_3d[:, 0],
    y=vectors_3d[:, 1],
    z=vectors_3d[:, 2],
    mode="markers+text",
    text=labels,
    textposition="top center",
    marker=dict(
        size=5,
        color=colors,  
        opacity=0.8
    )
)])


fig.update_layout(
    title=f"Visualização 3D de {len(labels)} Palavras (Reduzidas de 32D para 3D com t-SNE)",
    scene=dict(
        xaxis_title='Componente t-SNE 1',
        yaxis_title='Componente t-SNE 2',
        zaxis_title='Componente t-SNE 3'
    ),
    margin=dict(r=20, l=10, b=10, t=40)
)

fig.show()

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 8:</b><br>
  Embora o modelo desenvolvido seja simples e apresente limitações para distinguir claramente contextos positivos, neutros e negativos, é possível identificar algum padrão ou indício de formação de clusters nos dados?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

In [ ]:
# extração da saída intermediária da camada LSTM
model_input = model_lstm.layers[0].input 
lstm_layer_output = model_lstm.layers[2].output

feature_extractor_lstm = Model(inputs=model_input, outputs=lstm_layer_output)

# seleciona as amostras aleatórias do conjunto de teste
num_samples_to_plot = 1000
np.random.seed(42)
indices = np.random.choice(len(x_test), num_samples_to_plot, replace=False)
x_test_sample = x_test[indices]
y_test_sample = y_test[indices] 

# extração de features (saída da LSTM)
lstm_features = feature_extractor_lstm.predict(x_test_sample, verbose=0)
print(f"Shape das features extraídas: {lstm_features.shape}")

# realiza previsões para essas amostras
y_pred_probs_lstm = model_lstm.predict(x_test_sample, verbose=0)
y_pred_lstm = (y_pred_probs_lstm > 0.5).astype(int).flatten()

# define as cores para cada tipo de caso
colors = []
for true, pred in zip(y_test_sample, y_pred_lstm):
    if true == 0 and pred == 0:
        colors.append("blue")       
    elif true == 1 and pred == 1:
        colors.append("green")     
    elif true == 0 and pred == 1:
        colors.append("red")        
    else:
        colors.append("orange")     

hover_labels = [
    f"Verdadeiro: {'Positivo' if t == 1 else 'Negativo'} | "
    f"Previsto: {'Positivo' if p == 1 else 'Negativo'}"
    for t, p in zip(y_test_sample, y_pred_lstm)
]

# reduz as features de 64D para 3D com t-SNE
tsne_3d = TSNE(n_components=3, random_state=42, perplexity=30, max_iter=1000)
features_3d = tsne_3d.fit_transform(lstm_features)

In [ ]:
# plota o gráfico 3D interativo
fig = go.Figure(data=[go.Scatter3d(
    x=features_3d[:, 0],
    y=features_3d[:, 1],
    z=features_3d[:, 2],
    mode='markers',
    marker=dict(
        size=5,
        color=colors,
        opacity=0.8
    ),
    text=hover_labels,
    hoverinfo='text'
)])

fig.update_layout(
    title="Visualização 3D — Separação de Sentimentos (Saída da LSTM)",
    scene=dict(
        xaxis_title='Componente t-SNE 1',
        yaxis_title='Componente t-SNE 2',
        zaxis_title='Componente t-SNE 3'
    ),
    margin=dict(r=20, l=10, b=10, t=40),
    legend_title_text="Classificação"
)

fig.show()

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 9:</b><br>
  A projeção t-SNE mostra uma certa organização dos dados, mas também sobreposições entre classes.
Que conclusões podemos tirar disso sobre a capacidade da LSTM em capturar as fronteiras entre sentimentos positivos, neutros e negativos?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 10:</b><br>
    Comparando os modelos com camada simples de RNN e camadas de LSTM, qual a sua conclusão sobre os desempenhos de cada um nessa tarefa? Quais pontos positivos e negativos são possíveis de observar na capacidade de predição do segundo modelo?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Para Refletir:</b><br>
    Diante das limitações do modelo simples usado (LSTM) na captura de contexto mais complexo, por exemplo, frases ambíguas ou irônicas, o que você sugeriria para melhorar essa compreensão de contexto no próximo passo do projeto? Pense em: (a) arquitetura/modelo (por exemplo, outro tipo de rede), (b) pré-processamento ou enriquecimento de dados (por exemplo, inclusão de conhecimento externo, embeddings mais ricos), (c) técnicas de treinamento (exemplo: fine-tuning, regularização, adversarial training). Justifique suas sugestões modificando no código acima e comprovando que ocorreu melhorias nos modelos treinados.
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

### Links Úteis

- [Measure and Improve Robustness in NLP Models: A Survey](https://aclanthology.org/2022.naacl-main.339.pdf)
- [Natural Language Processing with Commonsense Knowledge: A Survey](https://arxiv.org/pdf/2108.04674)
- [Improving the Accuracy and Effectiveness of Text Classification Based on the Integration of the Bert Model and a Recurrent Neural Network](https://www.mdpi.com/2076-3417/14/18/8388)
- [A robust hybrid approach with product context-aware learning and explainable AI for sentiment analysis in Amazon user reviews](https://link.springer.com/article/10.1007/s10660-024-09896-5)